# 3 — Advanced model: LSTM trained in simulation (BPTT)

Self-contained notebook for the **advanced approach** of the ANN assignment.

## Motivation

The simple NARX-MLP (trained in `final/2_narx_simple.ipynb`) is excellent at one-step
**prediction** (~0.39°) but degrades in **simulation** (~7°). The reason is the
**exposure bias** of teacher-forced training: at training time the network always sees
true past angles, so it never learns to recover from its own mistakes during free-run.

In the exploration folder we tried two ways to close this gap:
- `exploration/3_advanced_NOE-vs-LSTM32_T-sweep.ipynb` — same NARX-MLP retrained in
  simulation mode (NOE) and a first LSTM with hidden=32. The LSTM(T=50) reached
  val SIM 0.0587 rad (3.36°), clearly beating the simulation-trained NOE.
- `exploration/4_advanced_NOE-stabilize-and-LSTM64.ipynb` — bigger LSTM (hidden=64,
  T=50) → val SIM 0.0449 rad (2.57°).

This notebook **pushes the LSTM further** by exploring capacity (width × depth) on three
variants, picks the best one as our final advanced model, and writes the submission files.

## What this notebook does
1. Loads the processed signals produced by `final/1_data_prep.ipynb`.
2. Defines a stacked LSTM (1 or 2 layers) with a `(sin θ, cos θ)` output head.
3. Trains 3 variants in simulation mode with BPTT (warm-up of 20 true samples, then a
   T=50 rollout where the network feeds its own predictions back as input):
     - hidden=128, 1 layer
     - hidden=128, 2 layers
     - hidden=64,  2 layers
4. Compares all three (plus the reference numbers from the simple NARX and the
   exploration LSTMs h=32 and h=64) and selects the winner on val **simulation** RMSE.
5. Saves the winner to `artifacts/best_advanced.pt` and writes
   `submissions/lstm-v3-{prediction,simulation}-submission-file.npz`.

## Training recipe (all 3 variants)
- Input per time step: `[ũ_k, sin θ_k, cos θ_k]` (`ũ = u/3`).
- Warm-up `N_init = 20` steps with teacher forcing → settles `(h, c)` from a real history.
- Rollout `T = 50` simulated steps where the LSTM uses **its own predictions** as
  `(sin, cos)` input; loss is computed on the T scored outputs.
- Adam, `lr=1e-3`, `weight_decay=1e-4`.
- ReduceLROnPlateau (factor 1/3, patience 5) on val SIM.
- **Gradient clipping** at norm 1.0 (critical for BPTT stability).
- **Unit-circle penalty** `λ_uc · mean((ŝ²+ĉ²−1)²)` with `λ_uc = 1e-2` (keeps the output
  on the sin/cos manifold).
- Early stopping on val SIM with patience 10, max 3500 iterations.


## 3.1 — Setup

In [1]:
import time, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

REPO_DIR  = Path('/Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/gym-unbalanced-disk-master/disc-benchmark-files')
WORK_DIR  = Path('/Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment')
ARTIFACTS = WORK_DIR / 'artifacts'
SUBMISSIONS = WORK_DIR / 'submissions'
SUBMISSIONS.mkdir(exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(2); np.random.seed(2)
print('device:', device)


device: mps


## 3.2 — Helpers and data

In [2]:
def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def rmse_from_sincos(s_pred, c_pred, s_true, c_true):
    th_p = np.arctan2(s_pred, c_pred)
    th_t = np.arctan2(s_true, c_true)
    return float(np.sqrt(np.mean(wrap_pi(th_p - th_t) ** 2)))

P = np.load(ARTIFACTS / 'step2_processed.npz')
u_tr_n, u_va_n = P['u_tr_n'], P['u_va_n']
sin_tr, cos_tr = P['sin_tr'], P['cos_tr']
sin_va, cos_va = P['sin_va'], P['cos_va']
U_SCALE = float(P['U_SCALE'])

u_tr_t   = torch.from_numpy(u_tr_n.astype(np.float32)).to(device)
sin_tr_t = torch.from_numpy(sin_tr.astype(np.float32)).to(device)
cos_tr_t = torch.from_numpy(cos_tr.astype(np.float32)).to(device)
u_va_t   = torch.from_numpy(u_va_n.astype(np.float32)).to(device)
sin_va_t = torch.from_numpy(sin_va.astype(np.float32)).to(device)
cos_va_t = torch.from_numpy(cos_va.astype(np.float32)).to(device)
print(f'Train: {len(u_tr_n)}   Val: {len(u_va_n)}')


Train: 28000   Val: 7000


## 3.3 — Stacked LSTM model

A `num_layers`-deep stack of `LSTMCell`s. The hidden state is `[h_l, c_l]` for each
layer `l`; the output is a linear head on the top-layer `h`.


In [3]:
class StackedLSTM(nn.Module):
    def __init__(self, hidden=128, num_layers=2, in_dim=3, out_dim=2):
        super().__init__()
        self.hidden = hidden
        self.num_layers = num_layers
        self.cells = nn.ModuleList([
            nn.LSTMCell(input_size=in_dim if i == 0 else hidden, hidden_size=hidden)
            for i in range(num_layers)
        ])
        self.head = nn.Linear(hidden, out_dim)

    def init_state(self, B, device):
        h = [torch.zeros(B, self.hidden, device=device) for _ in range(self.num_layers)]
        c = [torch.zeros(B, self.hidden, device=device) for _ in range(self.num_layers)]
        return h, c

    def step(self, x, h, c):
        out = x
        new_h, new_c = [], []
        for i, cell in enumerate(self.cells):
            hi, ci = cell(out, (h[i], c[i]))
            new_h.append(hi); new_c.append(ci)
            out = hi
        return out, new_h, new_c

N_INIT = 20

def unroll_window(model, u_win, sin_win, cos_win, n_init, T):
    """Warm-up n_init steps with teacher forcing, then T-1 more in simulation.
    Returns (pred_sin, pred_cos) each of shape (B, T) for time indices [n_init, n_init+T-1].
    """
    B = u_win.shape[0]
    h, c = model.init_state(B, u_win.device)
    for t in range(n_init):
        x = torch.stack([u_win[:, t], sin_win[:, t], cos_win[:, t]], dim=-1)
        out, h, c = model.step(x, h, c)
    last_pred = model.head(out)
    scored = [last_pred]
    for t in range(n_init, n_init + T - 1):
        x = torch.stack([u_win[:, t], last_pred[:, 0], last_pred[:, 1]], dim=-1)
        out, h, c = model.step(x, h, c)
        last_pred = model.head(out)
        scored.append(last_pred)
    pred = torch.stack(scored, dim=1)
    return pred[..., 0], pred[..., 1]

def sample_batch(B, T):
    L = N_INIT + T; Ntr = u_tr_t.shape[0]
    k0 = torch.randint(low=0, high=Ntr - L + 1, size=(B,), device=device)
    ar = torch.arange(L, device=device); idx = k0[:, None] + ar
    return u_tr_t[idx], sin_tr_t[idx], cos_tr_t[idx]

@torch.no_grad()
def eval_val_sim(model):
    model.eval()
    Nv = u_va_t.shape[0]; T_eff = Nv - N_INIT
    ps, pc = unroll_window(model, u_va_t.unsqueeze(0), sin_va_t.unsqueeze(0),
                            cos_va_t.unsqueeze(0), N_INIT, T_eff)
    ps = ps.squeeze(0).cpu().numpy(); pc = pc.squeeze(0).cpu().numpy()
    return rmse_from_sincos(ps, pc, sin_va[N_INIT:], cos_va[N_INIT:])

@torch.no_grad()
def eval_val_pred(model):
    """One-step prediction RMSE on val: feed the LSTM teacher-forced over the whole val
    sequence, score the output at each step against the next true (sin, cos)."""
    model.eval()
    Nv = u_va_t.shape[0]
    h, c = model.init_state(1, device)
    out = None
    preds_s, preds_c = [], []
    for t in range(Nv - 1):
        x = torch.stack([u_va_t[t:t+1], sin_va_t[t:t+1], cos_va_t[t:t+1]], dim=-1)
        out, h, c = model.step(x, h, c)
        y = model.head(out)
        preds_s.append(y[0, 0].item()); preds_c.append(y[0, 1].item())
    ps = np.array(preds_s); pc = np.array(preds_c)
    return rmse_from_sincos(ps[N_INIT-1:], pc[N_INIT-1:], sin_va[N_INIT:], cos_va[N_INIT:])


## 3.4 — Training routine

Same recipe for all variants. Returns the trained model plus its validation metrics.


In [4]:
def train_lstm(hidden, num_layers, T_horizon=50, max_iters=3500, B=64,
               lr=1e-3, wd=1e-4, lambda_uc=1e-2,
               eval_every=100, patience=10, verbose=True):
    model = StackedLSTM(hidden=hidden, num_layers=num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=1/3, patience=5)
    best_sim = float('inf'); best_state = copy.deepcopy(model.state_dict()); bad = 0
    history = {'iter': [], 'train_loss': [], 'val_sim': []}
    t0 = time.time()
    for it in range(1, max_iters + 1):
        model.train()
        u_w, s_w, c_w = sample_batch(B, T_horizon)
        ps, pc = unroll_window(model, u_w, s_w, c_w, N_INIT, T_horizon)
        s_t = s_w[:, N_INIT:N_INIT + T_horizon]; c_t = c_w[:, N_INIT:N_INIT + T_horizon]
        mse = ((ps - s_t) ** 2 + (pc - c_t) ** 2).mean()
        uc  = ((ps ** 2 + pc ** 2 - 1.0) ** 2).mean()
        loss = mse + lambda_uc * uc
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if it % eval_every == 0:
            v = eval_val_sim(model); sched.step(v)
            history['iter'].append(it); history['train_loss'].append(float(loss.item())); history['val_sim'].append(v)
            if v < best_sim - 1e-6:
                best_sim = v; best_state = copy.deepcopy(model.state_dict()); bad = 0
            else:
                bad += 1
            if verbose:
                print(f'  it={it:>4}  loss={loss.item():.4e}  val_sim={v:.4f}  best={best_sim:.4f}  bad={bad}')
            if bad >= patience:
                if verbose: print(f'  early stop at iter {it}')
                break
    model.load_state_dict(best_state)
    return {
        'name': f'LSTM h={hidden}, L={num_layers}',
        'hidden': hidden, 'num_layers': num_layers,
        'state_dict': best_state,
        'val_pred': eval_val_pred(model),
        'val_sim':  eval_val_sim(model),
        'history':  history,
        'time_s':   time.time() - t0,
    }


## 3.5 — Train the 3 LSTM variants

Three configurations:
- **h=128, L=1**: wider only (vs. the exploration baseline h=64, L=1).
- **h=128, L=2**: wider **and** deeper.
- **h=64,  L=2**: deeper only — control to see whether width or depth is what matters.

Total training time on Apple Silicon (MPS): ~10–15 minutes.


In [5]:
variants = [
    {'hidden': 128, 'num_layers': 1},
    {'hidden': 128, 'num_layers': 2},
    {'hidden': 64,  'num_layers': 2},
]
results = []
for v in variants:
    print(f'\n--- Training {v["hidden"]} hidden, {v["num_layers"]} layer(s) ---')
    r = train_lstm(**v)
    print(f'  -> val PRED = {r["val_pred"]:.4f}   val SIM = {r["val_sim"]:.4f} rad ({np.rad2deg(r["val_sim"]):.2f}\u00b0)   [{r["time_s"]:.1f}s]')
    results.append(r)



--- Training 128 hidden, 1 layer(s) ---


  it= 100  loss=3.0364e-02  val_sim=0.2531  best=0.2531  bad=0


  it= 200  loss=7.0986e-03  val_sim=0.2456  best=0.2456  bad=0


  it= 300  loss=1.7728e-03  val_sim=0.2094  best=0.2094  bad=0


  it= 400  loss=1.9431e-03  val_sim=0.0881  best=0.0881  bad=0


...

  it=3200  loss=5.3085e-04  val_sim=0.0467  best=0.0467  bad=0


  it=3300  loss=6.7813e-04  val_sim=0.0481  best=0.0467  bad=1


  it=3400  loss=7.5036e-04  val_sim=0.0539  best=0.0467  bad=2


  it=3500  loss=7.5782e-04  val_sim=0.0448  best=0.0448  bad=0


  -> val PRED = 0.0291   val SIM = 0.0448 rad (2.57°)   [295.8s]


## 3.6 — Comparison table

Numbers from the reference baselines and earlier explorations:
- **Simple NARX (8,8)** from `final/2_narx_simple.ipynb`.
- **LSTM h=32** from `exploration/3_advanced_NOE-vs-LSTM32_T-sweep.ipynb`.
- **LSTM h=64** from `exploration/4_advanced_NOE-stabilize-and-LSTM64.ipynb`.
- Repo baselines (Linear ARX, Good NN, lower bound) from the public README.


In [6]:
REF = [
    {'name': 'NARX-MLP (8,8) — final/2',         'val_pred': 0.0068, 'val_sim': 0.1221},
    {'name': 'LSTM h=32, L=1 — exploration/3',   'val_pred': 0.0319, 'val_sim': 0.0587},
    {'name': 'LSTM h=64, L=1 — exploration/4',   'val_pred': 0.0243, 'val_sim': 0.0449},
]
all_rows = REF + [
    {'name': r['name'], 'val_pred': r['val_pred'], 'val_sim': r['val_sim']} for r in results
]
print(f'{"model":<40} {"val_PRED [rad]":>15} {"val_SIM [rad]":>14} {"val_SIM [deg]":>14}')
print('-' * 90)
for r in all_rows:
    print(f'{r["name"]:<40} {r["val_pred"]:>15.4f} {r["val_sim"]:>14.4f} {np.rad2deg(r["val_sim"]):>14.2f}')
print()
print('Repo baselines:')
print(f'{"Good NN  (uses omega?)":<40} {0.00382:>15.4f} {0.0271:>14.4f} {1.55:>14.2f}')
print(f'{"Linear ARX":<40} {0.00665:>15.4f} {0.255:>14.4f} {14.6:>14.2f}')
print(f'{"Lower bound (sim)":<40} {"-":>15} {0.0195:>14.4f} {1.12:>14.2f}')


model                                     val_PRED [rad]  val_SIM [rad]  val_SIM [deg]
------------------------------------------------------------------------------------------
NARX-MLP (8,8) — final/2                          0.0068         0.1221           7.00
LSTM h=32, L=1 — exploration/3                    0.0319         0.0587           3.36
LSTM h=64, L=1 — exploration/4                    0.0243         0.0449           2.57
LSTM h=128, L=1                                   0.0243         0.0481           2.75
LSTM h=128, L=2                                   0.0266         0.0385           2.20
LSTM h=64, L=2                                    0.0291         0.0448           2.57

Repo baselines:
Good NN  (uses omega?)                            0.0038         0.0271           1.55
Linear ARX                                        0.0066         0.2550          14.60
Lower bound (sim)                                      -         0.0195           1.12


In [7]:
# Bar chart.
names = [r['name'] for r in all_rows]
sims  = [r['val_sim']  for r in all_rows]
preds = [r['val_pred'] for r in all_rows]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(len(names)), preds, color='steelblue')
ax[0].axhline(0.00382, color='green', ls='--', lw=0.8, label='Good NN pred')
ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax[0].set_ylabel('val PRED [rad]'); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)
ax[1].bar(range(len(names)), sims, color='darkorange')
ax[1].axhline(0.0271, color='green', ls='--', lw=0.8, label='Good NN sim')
ax[1].axhline(0.0195, color='red',   ls='--', lw=0.8, label='Lower bound')
ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax[1].set_ylabel('val SIM [rad]'); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


/var/folders/fj/76zlgq8s5nx12_qh_6tz449h0000gn/T/ipykernel_22366/3032118889.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3.7 — Pick the winner and save

The winner is the variant with the lowest val SIM RMSE among the three trained here. We
save its weights to `artifacts/best_advanced.pt` together with all the metadata needed
to rebuild the model from scratch.


In [8]:
winner = min(results, key=lambda r: r['val_sim'])
print(f'Winner among this notebook\'s variants: {winner["name"]}')
print(f'  val PRED = {winner["val_pred"]:.4f} rad')
print(f'  val SIM  = {winner["val_sim"]:.4f} rad ({np.rad2deg(winner["val_sim"]):.2f}\u00b0)')

# Rebuild model and save weights with metadata.
win_model = StackedLSTM(hidden=winner['hidden'], num_layers=winner['num_layers']).to(device)
win_model.load_state_dict(winner['state_dict']); win_model.eval()

torch.save({
    'family':       'LSTM',
    'architecture': 'StackedLSTM',
    'hidden':       winner['hidden'],
    'num_layers':   winner['num_layers'],
    'n_init':       N_INIT,
    'state_dict':   winner['state_dict'],
    'val_pred':     winner['val_pred'],
    'val_sim':      winner['val_sim'],
    'U_SCALE':      U_SCALE,
}, ARTIFACTS / 'best_advanced.pt')
print(f'Saved {ARTIFACTS / "best_advanced.pt"}')


Winner among this notebook's variants: LSTM h=128, L=2
  val PRED = 0.0266 rad
  val SIM  = 0.0385 rad (2.20°)
Saved /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/artifacts/best_advanced.pt


## 3.8 — Free-run plot of the winner on the validation set

In [9]:
Nv = u_va_t.shape[0]
with torch.no_grad():
    ps, pc = unroll_window(win_model, u_va_t.unsqueeze(0), sin_va_t.unsqueeze(0),
                             cos_va_t.unsqueeze(0), N_INIT, Nv - N_INIT)
th_best = np.arctan2(ps.squeeze(0).cpu().numpy(), pc.squeeze(0).cpu().numpy())
th_meas = np.arctan2(sin_va, cos_va)[N_INIT:]

n_show = 400
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(th_meas[:n_show], label='measured', lw=1.1)
ax.plot(th_best[:n_show], label=f'{winner["name"]} free-run', lw=1.0, alpha=0.85)
ax.set_xlabel('samples after N_INIT'); ax.set_ylabel(r'$\theta$ [rad]'); ax.grid(alpha=0.3); ax.legend()
ax.set_title(f'Validation free-run, first {n_show} samples — {winner["name"]}')
plt.tight_layout(); plt.show()


/var/folders/fj/76zlgq8s5nx12_qh_6tz449h0000gn/T/ipykernel_22366/196385039.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3.9 — Generate the **prediction** submission

For the prediction task we have 15 past samples of `(u, θ)` per test point. We feed all
15 through the LSTM (no rollout, just teacher forcing on `(sin, cos)`) and read the last
output as $\hat\theta_k$.


In [10]:
test_pred = np.load(REPO_DIR / 'hidden-test-prediction-submission-file.npz')
upast  = test_pred['upast']; thpast = test_pred['thpast']; Ntp = upast.shape[0]
u_n = (upast / U_SCALE).astype(np.float32)
s_n = np.sin(thpast).astype(np.float32); c_n = np.cos(thpast).astype(np.float32)
Up = torch.from_numpy(u_n).to(device)
Sp = torch.from_numpy(s_n).to(device); Cp = torch.from_numpy(c_n).to(device)
with torch.no_grad():
    h, c = win_model.init_state(Ntp, device)
    out = None
    for t in range(upast.shape[1]):
        x = torch.stack([Up[:, t], Sp[:, t], Cp[:, t]], dim=-1)
        out, h, c = win_model.step(x, h, c)
    Y = win_model.head(out).cpu().numpy()
th_pred_test = np.arctan2(Y[:, 0], Y[:, 1])
out_pred = SUBMISSIONS / 'lstm-v3-prediction-submission-file.npz'
np.savez(out_pred, upast=upast, thpast=thpast, thnow=th_pred_test)
print(f'Saved {out_pred}    range: [{th_pred_test.min():.3f}, {th_pred_test.max():.3f}]')


Saved /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/submissions/lstm-v3-prediction-submission-file.npz    range: [-1.607, 1.670]


## 3.10 — Generate the **simulation** submission

Warm up the LSTM on the first 50 true samples (provided), then free-run for 4950 more
samples using the test input sequence.


In [11]:
test_sim = np.load(REPO_DIR / 'hidden-test-simulation-submission-file.npz')
u_sim_raw = test_sim['u']; th_sim_seed = test_sim['th'].copy()
u_sim_n = (u_sim_raw / U_SCALE).astype(np.float32)
sin_seed = np.sin(th_sim_seed).astype(np.float32)
cos_seed = np.cos(th_sim_seed).astype(np.float32)
skip_sim = 50
Nsim = len(u_sim_n)
Up = torch.from_numpy(u_sim_n).to(device).unsqueeze(0)
Sp = torch.from_numpy(sin_seed).to(device).unsqueeze(0)
Cp = torch.from_numpy(cos_seed).to(device).unsqueeze(0)
with torch.no_grad():
    h, c = win_model.init_state(1, device)
    out = None
    for t in range(skip_sim):
        x = torch.stack([Up[:, t], Sp[:, t], Cp[:, t]], dim=-1)
        out, h, c = win_model.step(x, h, c)
    last_pred = win_model.head(out)
    out_s, out_c = [], []
    for t in range(skip_sim, Nsim):
        x = torch.stack([Up[:, t], last_pred[:, 0], last_pred[:, 1]], dim=-1)
        out, h, c = win_model.step(x, h, c)
        last_pred = win_model.head(out)
        out_s.append(last_pred[0, 0].item()); out_c.append(last_pred[0, 1].item())
th_after = np.arctan2(np.array(out_s), np.array(out_c))
th_sim_full = np.concatenate([th_sim_seed[:skip_sim], th_after])
out_sim = SUBMISSIONS / 'lstm-v3-simulation-submission-file.npz'
np.savez(out_sim, th=th_sim_full, u=u_sim_raw)
print(f'Saved {out_sim}    range: [{th_sim_full.min():.3f}, {th_sim_full.max():.3f}]')


Saved /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/submissions/lstm-v3-simulation-submission-file.npz    range: [-1.769, 1.889]


## 3.11 — Format check with the official `submission-file-checker.py`

Validates only the **format** (keys, shape). The RMSE printed is meaningless because the
solution file is the zero-filled hidden test file; the real RMSE on the hidden ground
truth can only be computed by the lecturers.


In [12]:
import subprocess, sys, os
checker = REPO_DIR / 'submission-file-checker.py'
for sub, sol in [
    (out_pred, REPO_DIR / 'hidden-test-prediction-submission-file.npz'),
    (out_sim,  REPO_DIR / 'hidden-test-simulation-submission-file.npz'),
]:
    print(f'\n--- checking {sub.name} ---')
    res = subprocess.run([sys.executable, str(checker), str(sub), str(sol)],
                         capture_output=True, text=True,
                         env={**os.environ, 'MPLBACKEND': 'Agg'})
    print('STDOUT:', res.stdout.strip())
    if res.returncode != 0:
        print('STDERR:', res.stderr.strip())



--- checking lstm-v3-prediction-submission-file.npz ---


STDOUT: ################ Prediction Result ##################
RMS= 0.5467 radians
RMS= 31.322 degrees
NRMS= inf%

--- checking lstm-v3-simulation-submission-file.npz ---


STDOUT: ################ Simulation Result ##################
RMS= 0.5433 radians
RMS= 31.127 degrees
NRMS= 1650.33%


## 3.12 — Summary

In [13]:
print('==============  LSTM (advanced model)  ==============')
print(f'Architecture : StackedLSTM hidden={winner["hidden"]}, layers={winner["num_layers"]}, n_init={N_INIT}')
print(f'Val PRED RMSE_theta = {winner["val_pred"]:.4f} rad   ({np.rad2deg(winner["val_pred"]):.3f}\u00b0)')
print(f'Val SIM  RMSE_theta = {winner["val_sim"]:.4f} rad   ({np.rad2deg(winner["val_sim"]):.3f}\u00b0)')
print(f'\nFiles saved:')
print(f'  Model:        {ARTIFACTS / "best_advanced.pt"}')
for p in sorted(SUBMISSIONS.glob('lstm-v3-*.npz')):
    print(f'  Submission:   {p}')


==============  LSTM (advanced model)  ==============
Architecture : StackedLSTM hidden=128, layers=2, n_init=20
Val PRED RMSE_theta = 0.0266 rad   (1.527°)
Val SIM  RMSE_theta = 0.0385 rad   (2.203°)

Files saved:
  Model:        /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/artifacts/best_advanced.pt
  Submission:   /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/submissions/lstm-v3-prediction-submission-file.npz
  Submission:   /Users/davidedeleonardis/Documents/TU-e/1 anno/Q4/Machine learning for systems and control/Assigment/ML4S&C-assignment/submissions/lstm-v3-simulation-submission-file.npz


---
**End of notebook 3.**

This is the advanced model deliverable. The simple model is in `2_narx_simple.ipynb`.
